# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

In [31]:
'There are 17 marketing campaigns in the dataset.'

'There are 17 marketing campaigns in the dataset.'

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [2]:
import pandas as pd

In [3]:
bank_full = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [4]:
bank_full.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



There are 3 features that include missing values
1) Categorical - based on the "unknown" value
2) pdays - based on the "999" value
3) poutcome - based on the "nonexistent"

There are 11 features that may need to be coerced to a different data type. The reason is because this may help with the relationship mathematically per column.
1) y
2) contact
3) default
4) housing
5) loan
6) education
7) month
8) day_of_week
9) job
10) marital
11) poutcome

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [8]:
bank_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

The Business Objective is to increase term deposit subscriptions by identifying banking clients most likely to respond positively to direct telemarketing campaigns (y = 'yes'), enabling more effective customer targeting, higher conversion rates, and optimized marketing costs and resources.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

In [10]:
# Problem 5: Select and encode features and target
bank_features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
X = pd.get_dummies(bank_full[bank_features], drop_first=True)
y = bank_full['y'].map({'yes': 1, 'no': 0})

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [11]:
# Split the prepared features (X) and target (y) into 80% train and 20% test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Verify the shapes of the resulting splits
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"y_test distribution:\n{y_test.value_counts(normalize=True)}")

X_train shape: (32950, 28)
X_test shape: (8238, 28)
y_train distribution:
y
0    0.887344
1    0.112656
Name: proportion, dtype: float64
y_test distribution:
y
0    0.887351
1    0.112649
Name: proportion, dtype: float64


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

The baseline perfomance that our classifier should aim to beat is 88.7%.  The 88.7% covers the clients that did not subscribe to a term deposit (y='no').

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [12]:
from sklearn.linear_model import LogisticRegression

In [14]:
# 1. Initialize the Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# 2. Fit the model directly on your already-encoded training data
log_reg.fit(X_train, y_train)

# 3. Generate predictions
y_train_pred = log_reg.predict(X_train)
y_test_pred = log_reg.predict(X_test)

### Problem 9: Score the Model

What is the accuracy of your model?

In [15]:
from sklearn.metrics import accuracy_score

# Calculate training and test accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy:.2%})")
print(f"Test Accuracy:     {test_accuracy:.4f} ({test_accuracy:.2%})")

Training Accuracy: 0.8873 (88.73%)
Test Accuracy:     0.8874 (88.74%)


### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [19]:
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [20]:
# 1. Define the models with default settings
models = {
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'SVM': SVC()
}

# 2. Train, time, and score each model
results = []

for name, model in models.items():
    # Record start time and fit the model
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Calculate train and test accuracy
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    
    results.append({
        'Model': name,
        'Train Time': round(train_time, 4),
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy': round(test_acc, 4)
    })

# 3. Display results in a DataFrame
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              Model  Train Time  Train Accuracy  Test Accuracy
Logistic Regression      0.1307          0.8873         0.8874
                KNN      0.0040          0.8914         0.8775
      Decision Tree      0.0613          0.9171         0.8632
                SVM      5.8405          0.8873         0.8874


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.metrics import f1_score

In [23]:
#Prepare the full pre-call dataset (drop 'duration' to avoid data leakage)
X_full = pd.get_dummies(bank_full.drop(columns=['y', 'duration']), drop_first=True)
y_full = bank_full['y'].map({'yes': 1, 'no': 0})

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

In [25]:
# Define models (wrapped in scaling pipelines where needed) and parameter grids
model_grids = {
    'Logistic Regression': (
        Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
        {'clf__C': [0.01, 0.1, 1, 10], 'clf__class_weight': [None, 'balanced']}
    ),
    'KNN': (
        Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]),
        {'clf__n_neighbors': list((5, 15, 25)), 'clf__weights': ['uniform', 'distance']}
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {'max_depth': list((3, 5, 7, 10)), 'min_samples_leaf': list((10, 25, 50)), 'class_weight': [None, 'balanced']}
    ),
    'SVM': (
        Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
        {'clf__C': [0.1, 1, 10], 'clf__kernel': ['rbf']}
    )
}

In [30]:
# Run GridSearchCV optimizing for ROC-AUC and compare results
tuned_results = []
best_estimators = {}

for name, (estimator, param_grid) in model_grids.items():
    start_time = time.time()
    
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=3,
        n_jobs=-1
    )
    grid.fit(X_train_f, y_train_f)
    fit_time = time.time() - start_time
    
    best_model = grid.best_estimator_
    best_estimators[name] = best_model
    
    y_test_pred = best_model.predict(X_test_f)
    y_test_prob = best_model.predict_proba(X_test_f)[:, 1]
    
    tuned_results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'Grid Search Time (s)': round(fit_time, 2),
        'Test Accuracy': round(accuracy_score(y_test_f, y_test_pred), 4),
        'Test ROC-AUC': round(roc_auc_score(y_test_f, y_test_prob), 4),
        'Test F1 (Yes)': round(f1_score(y_test_f, y_test_pred), 4)
    })

tuned_df = pd.DataFrame(tuned_results)
tuned_df

/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

,Model,Best Params,Grid Search Time (s),Test Accuracy,Test ROC-AUC,Test F1 (Yes)
0,Logistic Regression,"{'clf__C': 1, 'clf__class_weight': 'balanced'}",0.44,0.8348,0.8009,0.4677
1,KNN,"{'clf__n_neighbors': 25, 'clf__weights': 'unif...",2.70,0.9014,0.7732,0.3504
2,Decision Tree,"{'class_weight': None, 'max_depth': 7, 'min_sa...",0.77,0.9011,0.8060,0.4098
3,SVM,"{'clf__C': 0.1, 'clf__kernel': 'rbf'}",225.33,0.8994,0.7329,0.2747


##### Questions